# 01 · Limpieza, transformación y unión de las fuentes

**Proyecto Final — Álvaro Velarde**

Este notebook toma las **diez tablas en bruto** de las dos fuentes y produce una única tabla analítica lista para el estudio estadístico y para el dashboard.

Las fuentes:

| Fuente | Origen | Contenido |
|---|---|---|
| Olist | Marketplace brasileño de comercio electrónico | 9 tablas: pedidos, artículos, pagos, reseñas, clientes, vendedores, productos, geolocalización y traducción de categorías |
| Estados de Brasil | Datos del instituto estadístico brasileño (IBGE) | Población, PIB por habitante, pobreza, superficie y coordenadas de los 27 estados, más el índice de desarrollo humano |

El objetivo del análisis es responder a una pregunta de negocio: **¿qué le está costando a Olist sus reseñas de una estrella?**

## Índice
1. Carga de las fuentes
2. Auditoría inicial
3. Preparar las tablas antes de unir
4. Construcción de la tabla analítica
5. Columnas derivadas
6. Verificación final
7. Guardado

## 1. Carga de las fuentes

In [3]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
from limpieza import (RegistroLimpieza, agregar_pagos, consolidar_geolocalizacion,
                      distancia_haversine, resolver_resenas_duplicadas, resumen_nulos)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RUTA = Path.cwd().parent / "data" / "raw"


def leer_del_zip(ruta_zip, nombre_csv, **kwargs):
    """
    Lee un CSV directamente desde dentro de un archivo comprimido.

    Se trabaja sobre los ZIP tal y como se descargaron en lugar de
    descomprimirlos: los nueve CSV de Olist ocupan 126 MB sueltos y solo 44 MB
    comprimidos. Así el repositorio guarda las fuentes exactamente como
    llegaron, sin alterarlas y sin ocupar de más.
    """
    with zipfile.ZipFile(ruta_zip) as z:
        with z.open(nombre_csv) as f:
            return pd.read_csv(f, **kwargs)


ZIP_OLIST  = RUTA / "olist_brazilian_ecommerce.zip"
ZIP_BRASIL = RUTA / "brasil_estados_ibge.zip"

# Fuente 1: Olist
pedidos     = leer_del_zip(ZIP_OLIST, "olist_orders_dataset.csv")
articulos   = leer_del_zip(ZIP_OLIST, "olist_order_items_dataset.csv")
pagos       = leer_del_zip(ZIP_OLIST, "olist_order_payments_dataset.csv")
resenas     = leer_del_zip(ZIP_OLIST, "olist_order_reviews_dataset.csv")
clientes    = leer_del_zip(ZIP_OLIST, "olist_customers_dataset.csv")
vendedores  = leer_del_zip(ZIP_OLIST, "olist_sellers_dataset.csv")
productos   = leer_del_zip(ZIP_OLIST, "olist_products_dataset.csv")
geo         = leer_del_zip(ZIP_OLIST, "olist_geolocation_dataset.csv")
categorias  = leer_del_zip(ZIP_OLIST, "product_category_name_translation.csv")

# Fuente 2: contexto socioeconómico de los estados
estados = leer_del_zip(ZIP_BRASIL, "states.csv")
idh     = leer_del_zip(ZIP_BRASIL, "hdi.csv")

tablas = {
    "pedidos": pedidos, "articulos": articulos, "pagos": pagos, "resenas": resenas,
    "clientes": clientes, "vendedores": vendedores, "productos": productos,
    "geolocalizacion": geo, "categorias": categorias,
    "estados": estados, "idh": idh,
}

for nombre, tabla in tablas.items():
    print(f"{nombre:<16} {tabla.shape[0]:>9,} filas x {tabla.shape[1]:>2} columnas")
print(f"\n{'TOTAL':<16} {sum(t.shape[0] for t in tablas.values()):>9,} filas en bruto")

pedidos             99,441 filas x  8 columnas
articulos          112,650 filas x  7 columnas
pagos              103,886 filas x  5 columnas
resenas             99,224 filas x  7 columnas
clientes            99,441 filas x  5 columnas
vendedores           3,095 filas x  4 columnas
productos           32,951 filas x  9 columnas
geolocalizacion  1,000,163 filas x  5 columnas
categorias              71 filas x  2 columnas
estados                 27 filas x 13 columnas
idh                     27 filas x 17 columnas

TOTAL            1,550,976 filas en bruto


## 2. Auditoría inicial

Antes de unir nada hay que entender el **grano** de cada tabla, es decir, qué representa una fila. Es la decisión más importante del proyecto: si se une una tabla con grano más fino sin agregarla antes, las filas se multiplican y todos los importes quedan inflados.

In [5]:
print("¿Cuántas filas tiene cada tabla por pedido?\n")
for nombre, tabla in [("articulos", articulos), ("pagos", pagos), ("resenas", resenas)]:
    conteo = tabla.groupby("order_id").size()
    print(f"{nombre:<12} media {conteo.mean():.2f} | máximo {conteo.max():>2} | "
          f"pedidos con más de uno: {(conteo > 1).sum():,}")

¿Cuántas filas tiene cada tabla por pedido?

articulos    media 1.14 | máximo 21 | pedidos con más de uno: 9,803
pagos        media 1.04 | máximo 29 | pedidos con más de uno: 2,961
resenas      media 1.01 | máximo  3 | pedidos con más de uno: 547


Tres tablas con grano más fino que el pedido:

- **Artículos**: hasta 21 en un mismo pedido. Este es el grano que nos interesa conservar, porque cada artículo tiene su propio vendedor, precio y flete.
- **Pagos**: hasta 29 movimientos en un pedido, por ejemplo combinando vales con tarjeta. **Hay que agregarlos antes de unir.**
- **Reseñas**: 547 pedidos con dos reseñas. **Hay que quedarse con una.**

In [7]:
print("VALORES AUSENTES\n")
for nombre in ["pedidos", "productos", "resenas"]:
    tabla = tablas[nombre]
    ausentes = resumen_nulos(tabla)
    if len(ausentes):
        print(f"-- {nombre}")
        print(ausentes.to_string())
        print()

VALORES AUSENTES

-- pedidos
                               nulos  porcentaje tipo
order_delivered_customer_date   2965        2.98  str
order_delivered_carrier_date    1783        1.79  str
order_approved_at                160        0.16  str

-- productos
                            nulos  porcentaje     tipo
product_category_name         610        1.85      str
product_name_lenght           610        1.85  float64
product_description_lenght    610        1.85  float64
product_photos_qty            610        1.85  float64
product_weight_g                2        0.01  float64
product_length_cm               2        0.01  float64
product_height_cm               2        0.01  float64
product_width_cm                2        0.01  float64

-- resenas
                        nulos  porcentaje tipo
review_comment_title    87656       88.34  str
review_comment_message  58247       58.70  str



In [8]:
print("ESTADO DE LOS PEDIDOS\n")
estado_pedidos = pedidos["order_status"].value_counts()
print(pd.DataFrame({
    "pedidos": estado_pedidos,
    "porcentaje": (estado_pedidos / len(pedidos) * 100).round(2),
}).to_string())

ESTADO DE LOS PEDIDOS

              pedidos  porcentaje
order_status                     
delivered       96478       97.02
shipped          1107        1.11
canceled          625        0.63
unavailable       609        0.61
invoiced          314        0.32
processing        301        0.30
created             5        0.01
approved            2        0.00


El 97% de los pedidos está entregado. Los 2.963 restantes están cancelados, en tránsito o pendientes, y son justamente los que tienen la fecha de entrega vacía: **los nulos de `order_delivered_customer_date` no son un error de captura, son pedidos que aún no habían llegado**. Esa distinción marca cómo tratarlos.

## 3. Preparar las tablas antes de unir

In [10]:
registro = RegistroLimpieza()

In [11]:
# --- Geolocalización: de un millón de filas a una por código postal ---
filas_antes = len(geo)
geo_limpia, fuera_de_brasil = consolidar_geolocalizacion(geo)

registro.anotar(
    "La tabla de geolocalización trae una fila por dirección: 1.000.163 filas para "
    "solo 19.015 códigos postales, con 261.831 duplicados exactos",
    filas_antes,
    "Agregar a una coordenada por código postal usando la mediana",
    "Unirla en crudo multiplicaría las filas de la tabla final por cientos. Se usa la "
    "mediana y no la media porque hay coordenadas erróneas que la media arrastraría.",
)

registro.anotar(
    "42 coordenadas caen fuera del territorio brasileño (una en pleno océano)",
    fuera_de_brasil,
    "Descartarlas antes de calcular la mediana por código postal",
    "Son errores de captura evidentes: Brasil está entre las latitudes -34 y 5.",
)

print(f"\nGeolocalización: {filas_antes:,} filas -> {len(geo_limpia):,} códigos postales")

[ 1] La tabla de geolocalización trae una fila por dirección: 1.000.163 filas para solo 19.015 códigos postales, con 261.831 duplicados exactos
     -> 1000163 | Agregar a una coordenada por código postal usando la mediana
[ 2] 42 coordenadas caen fuera del territorio brasileño (una en pleno océano)
     -> 42 | Descartarlas antes de calcular la mediana por código postal

Geolocalización: 1,000,163 filas -> 19,010 códigos postales


In [12]:
# --- Reseñas: una por pedido ---
duplicadas = int(resenas["order_id"].duplicated().sum())
resenas_limpias = resolver_resenas_duplicadas(resenas)

registro.anotar(
    f"{duplicadas} pedidos tienen dos reseñas, a veces con notas distintas",
    duplicadas,
    "Conservar la más reciente por fecha de creación",
    "Es la valoración definitiva del cliente una vez cerrada la incidencia. Quedarse "
    "con la primera daría por buena una opinión que el cliente ya cambió.",
)

print(f"\nReseñas: {len(resenas):,} -> {len(resenas_limpias):,}")

[ 3] 551 pedidos tienen dos reseñas, a veces con notas distintas
     -> 551 | Conservar la más reciente por fecha de creación

Reseñas: 99,224 -> 98,673


In [13]:
# --- Pagos: una fila por pedido ---
pagos_agregados = agregar_pagos(pagos)

registro.anotar(
    "Un pedido puede tener hasta 29 movimientos de pago (vales combinados con tarjeta)",
    len(pagos),
    "Agregar por pedido: importe total, plazos máximos y método principal",
    "El método principal se define como el que aporta más dinero al pedido, no el "
    "primero de la lista, para que la etiqueta signifique algo.",
)

print(f"\nPagos: {len(pagos):,} -> {len(pagos_agregados):,}")
print(pagos_agregados["metodo_pago"].value_counts().to_string())

[ 4] Un pedido puede tener hasta 29 movimientos de pago (vales combinados con tarjeta)
     -> 103886 | Agregar por pedido: importe total, plazos máximos y método principal

Pagos: 103,886 -> 99,440
metodo_pago
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
not_defined        3


In [14]:
# --- Productos: categorías ausentes y traducción ---
sin_categoria = int(productos["product_category_name"].isna().sum())

productos_limpios = productos.merge(categorias, on="product_category_name", how="left")

# Si no hay traducción se conserva el nombre original en portugués antes que
# perder la categoría entera.
productos_limpios["categoria"] = (
    productos_limpios["product_category_name_english"]
    .fillna(productos_limpios["product_category_name"])
    .fillna("sin_categoria")
)

registro.anotar(
    f"{sin_categoria} productos sin categoría asignada y nombres solo en portugués",
    sin_categoria,
    "Traducir con la tabla oficial y etiquetar los que faltan como 'sin_categoria'",
    "Etiquetarlos los mantiene visibles en los gráficos en lugar de hacerlos "
    "desaparecer silenciosamente al agrupar.",
)

print(f"\nCategorías distintas: {productos_limpios['categoria'].nunique()}")

[ 5] 610 productos sin categoría asignada y nombres solo en portugués
     -> 610 | Traducir con la tabla oficial y etiquetar los que faltan como 'sin_categoria'

Categorías distintas: 74


In [15]:
# --- Estados: unir el contexto socioeconómico con el desarrollo humano ---
estados_limpios = estados.merge(idh[["UF", "HDI 2017"]], on="UF", how="left")

estados_limpios = estados_limpios.rename(columns={
    "UF": "estado", "State": "estado_nombre", "Region": "region",
    "Population": "poblacion", "GDP": "pib_per_capita", "Poverty": "tasa_pobreza",
    "Demographic Density": "densidad", "HDI 2017": "idh",
    "Latitude": "estado_lat", "Longitude": "estado_lng",
})[["estado", "estado_nombre", "region", "poblacion", "pib_per_capita",
    "tasa_pobreza", "densidad", "idh", "estado_lat", "estado_lng"]]

registro.anotar(
    "La segunda fuente llega en dos archivos: indicadores generales por un lado e "
    "índice de desarrollo humano por otro",
    len(estados_limpios),
    "Unir ambos por el código de estado y renombrar las columnas al español",
    "El código de dos letras coincide exactamente con el estado del cliente en Olist, "
    "así que el cruce es directo y sin pérdidas.",
)

print(estados_limpios.head(3).to_string())

[ 6] La segunda fuente llega en dos archivos: indicadores generales por un lado e índice de desarrollo humano por otro
     -> 27 | Unir ambos por el código de estado y renombrar las columnas al español
  estado estado_nombre     region  poblacion  pib_per_capita  tasa_pobreza  densidad    idh  estado_lat  estado_lng
0     AC          Acre      North     881935        17201.95         0.189      5.37  0.719       -8.77      -70.55
1     AL       Alagoas  Northeast    3337357        15653.51         0.205    119.86  0.683       -9.62      -36.82
2     AM      Amazonas      North    4144597        22936.28         0.193      2.66  0.733       -3.47      -65.10


## 4. Construcción de la tabla analítica

El grano elegido es **una fila por artículo de pedido**. Un pedido con tres artículos son tres filas.

La alternativa sería agregar por pedido, pero eso destruiría información que necesitamos: cada artículo puede venir de un vendedor distinto, con su propio precio y su propio flete. Sin ese detalle no se puede analizar el rendimiento por vendedor ni por categoría, que es la mitad del proyecto.

In [17]:
# Se parte de los artículos y se van añadiendo las demás tablas
df = articulos.copy()
print(f"Punto de partida (artículos): {len(df):,} filas")

# Pedido y cliente
df = df.merge(pedidos, on="order_id", how="left")
df = df.merge(clientes, on="customer_id", how="left")
print(f"+ pedidos y clientes:        {len(df):,} filas")

# Vendedor y producto
df = df.merge(vendedores, on="seller_id", how="left")
df = df.merge(productos_limpios[["product_id", "categoria", "product_weight_g",
                                 "product_length_cm", "product_height_cm",
                                 "product_width_cm", "product_photos_qty"]],
              on="product_id", how="left")
print(f"+ vendedores y productos:    {len(df):,} filas")

# Reseña y pagos, ya agregados a una fila por pedido
df = df.merge(resenas_limpias[["order_id", "review_score", "review_creation_date"]],
              on="order_id", how="left")
df = df.merge(pagos_agregados, on="order_id", how="left")
print(f"+ reseñas y pagos:           {len(df):,} filas")

Punto de partida (artículos): 112,650 filas
+ pedidos y clientes:        112,650 filas
+ vendedores y productos:    112,650 filas
+ reseñas y pagos:           112,650 filas


**Las filas no se han movido de 112.650 en ningún paso.** Es la comprobación que confirma que las agregaciones previas funcionaron: si se hubieran unido los pagos o las reseñas sin agregar, aquí habría 130.000 o más.

In [19]:
# Coordenadas de cliente y vendedor a partir del código postal
df = df.merge(geo_limpia.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix",
                                         "lat": "cliente_lat", "lng": "cliente_lng"}),
              on="customer_zip_code_prefix", how="left")

df = df.merge(geo_limpia.rename(columns={"geolocation_zip_code_prefix": "seller_zip_code_prefix",
                                         "lat": "vendedor_lat", "lng": "vendedor_lng"}),
              on="seller_zip_code_prefix", how="left")

sin_coordenadas = int(df["cliente_lat"].isna().sum())
registro.anotar(
    "Algunos códigos postales de cliente y vendedor no aparecen en la tabla de "
    "geolocalización",
    sin_coordenadas,
    "Unir con left join y dejar las coordenadas a nulo",
    f"Son {sin_coordenadas / len(df):.1%} de las filas. Descartarlas por no tener "
    "coordenadas sacrificaría ventas reales por un dato que solo hace falta para el mapa.",
)

print(f"\nFilas tras las coordenadas: {len(df):,}")

[ 7] Algunos códigos postales de cliente y vendedor no aparecen en la tabla de geolocalización
     -> 303 | Unir con left join y dejar las coordenadas a nulo

Filas tras las coordenadas: 112,650


In [20]:
# Contexto socioeconómico del estado del cliente
df = df.merge(estados_limpios.rename(columns={"estado": "customer_state"}),
              on="customer_state", how="left")

# Y la región del vendedor, para analizar rutas entre regiones
df = df.merge(estados_limpios[["estado", "region"]].rename(
                  columns={"estado": "seller_state", "region": "region_vendedor"}),
              on="seller_state", how="left")

registro.anotar(
    "La tabla de Olist no contiene ninguna referencia demográfica ni económica",
    len(df),
    "Unir la segunda fuente por el código de estado del cliente y del vendedor",
    "Permite calcular penetración por habitante y comparar el comportamiento de los "
    "estados según su renta, que es la parte comercial del análisis.",
)

print(f"Filas tras el contexto socioeconómico: {len(df):,}")
print(f"Filas sin datos de estado: {int(df['poblacion'].isna().sum())}")

[ 8] La tabla de Olist no contiene ninguna referencia demográfica ni económica
     -> 112650 | Unir la segunda fuente por el código de estado del cliente y del vendedor
Filas tras el contexto socioeconómico: 112,650
Filas sin datos de estado: 0


## 5. Columnas derivadas

Aquí es donde los datos empiezan a poder responder preguntas. Ninguna de estas columnas existe en las fuentes: todas se calculan.

In [22]:
# Las fechas llegan como texto y hay que convertirlas antes de poder restarlas
columnas_fecha = ["order_purchase_timestamp", "order_approved_at",
                  "order_delivered_carrier_date", "order_delivered_customer_date",
                  "order_estimated_delivery_date", "shipping_limit_date",
                  "review_creation_date"]

for columna in columnas_fecha:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")

registro.anotar(
    "Las siete columnas de fecha llegan como texto y no permiten calcular plazos",
    len(df),
    "Convertir a datetime con errors='coerce'",
    "Con 'coerce' una fecha corrupta se convierte en nulo en lugar de detener el "
    "proceso entero, y después se puede contar cuántas fallaron.",
)

print(df[columnas_fecha].dtypes.to_string())

[ 9] Las siete columnas de fecha llegan como texto y no permiten calcular plazos
     -> 112650 | Convertir a datetime con errors='coerce'
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
shipping_limit_date              datetime64[us]
review_creation_date             datetime64[us]


In [23]:
# --- Plazos de entrega ---
df["dias_entrega_real"] = (df["order_delivered_customer_date"]
                           - df["order_purchase_timestamp"]).dt.days
df["dias_entrega_prometidos"] = (df["order_estimated_delivery_date"]
                                 - df["order_purchase_timestamp"]).dt.days
df["dias_retraso"] = (df["order_delivered_customer_date"]
                      - df["order_estimated_delivery_date"]).dt.days

# Un pedido "llega tarde" si supera la fecha que se le prometió al cliente.
# Se deja a nulo en los no entregados en lugar de marcarlos como puntuales:
# no sabemos si habrían llegado a tiempo.
df["llego_tarde"] = np.where(df["dias_retraso"].isna(), np.nan,
                             (df["dias_retraso"] > 0).astype(float))

df["dias_hasta_envio"] = (df["order_delivered_carrier_date"]
                          - df["order_purchase_timestamp"]).dt.days
df["horas_hasta_aprobacion"] = ((df["order_approved_at"]
                                 - df["order_purchase_timestamp"])
                                .dt.total_seconds() / 3600).round(2)

registro.anotar(
    "No existe ninguna medida de plazo: solo hay cinco marcas de tiempo sueltas",
    int(df["dias_entrega_real"].notna().sum()),
    "Calcular días de entrega reales y prometidos, días de retraso, días hasta la "
    "recogida por el transportista y horas hasta la aprobación del pago",
    "El retraso frente a lo prometido, y no el plazo absoluto, es lo que percibe el "
    "cliente: 20 días esperados no molestan, 5 días sobre 3 prometidos sí.",
)

print(df[["dias_entrega_real", "dias_entrega_prometidos", "dias_retraso"]].describe().round(1).to_string())

[10] No existe ninguna medida de plazo: solo hay cinco marcas de tiempo sueltas
     -> 110196 | Calcular días de entrega reales y prometidos, días de retraso, días hasta la recogida por el transportista y horas hasta la aprobación del pago
       dias_entrega_real  dias_entrega_prometidos  dias_retraso
count           110196.0                 112650.0      110196.0
mean                12.0                     23.5         -12.0
std                  9.5                      8.9          10.2
min                  0.0                      2.0        -147.0
25%                  6.0                     18.0         -17.0
50%                 10.0                     23.0         -13.0
75%                 15.0                     28.0          -7.0
max                209.0                    155.0         188.0


In [24]:
# --- Distancia entre vendedor y cliente ---
df["distancia_km"] = distancia_haversine(
    df["vendedor_lat"], df["vendedor_lng"], df["cliente_lat"], df["cliente_lng"]
).round(1)

df["mismo_estado"] = np.where(df["customer_state"] == df["seller_state"], "Sí", "No")

registro.anotar(
    "No hay forma directa de saber cuánto viaja cada pedido",
    int(df["distancia_km"].notna().sum()),
    "Calcular la distancia con la fórmula de Haversine entre las coordenadas del "
    "vendedor y las del cliente",
    "Restar coordenadas no vale: un grado de longitud mide distinto según la latitud. "
    "Haversine tiene en cuenta la curvatura terrestre.",
)

print(df["distancia_km"].describe().round(1).to_string())

[11] No hay forma directa de saber cuánto viaja cada pedido
     -> 112095 | Calcular la distancia con la fórmula de Haversine entre las coordenadas del vendedor y las del cliente
count    112095.0
mean        596.6
std         588.8
min           0.0
25%         183.1
50%         431.7
75%         791.8
max        3579.3


In [25]:
# --- Importes y desglose temporal ---
df["ingreso_linea"] = df["price"] + df["freight_value"]
df["peso_flete_pct"] = (df["freight_value"] / df["price"] * 100).round(2)

df["anio"] = df["order_purchase_timestamp"].dt.year
df["mes"] = df["order_purchase_timestamp"].dt.month
df["anio_mes"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["dia_semana"] = df["order_purchase_timestamp"].dt.day_name()
df["hora_compra"] = df["order_purchase_timestamp"].dt.hour

# Franja horaria: agrupar 24 horas en cuatro bloques hace los gráficos legibles
df["franja_horaria"] = pd.cut(
    df["hora_compra"], bins=[-1, 5, 11, 17, 23],
    labels=["Madrugada (0-5)", "Mañana (6-11)", "Tarde (12-17)", "Noche (18-23)"],
)

# Volumen del producto en litros, útil para explicar el coste del flete
df["volumen_cm3"] = (df["product_length_cm"] * df["product_height_cm"]
                     * df["product_width_cm"])

registro.anotar(
    "Faltan las métricas de negocio y el desglose temporal para el dashboard",
    len(df),
    "Crear ingreso por línea, peso del flete sobre el precio, año, mes, día de la "
    "semana, hora, franja horaria y volumen del producto",
    "Son las dimensiones por las que el dashboard va a permitir filtrar y segmentar.",
)

print(df["franja_horaria"].value_counts().sort_index().to_string())

[12] Faltan las métricas de negocio y el desglose temporal para el dashboard
     -> 112650 | Crear ingreso por línea, peso del flete sobre el precio, año, mes, día de la semana, hora, franja horaria y volumen del producto
franja_horaria
Madrugada (0-5)     5366
Mañana (6-11)      25225
Tarde (12-17)      43820
Noche (18-23)      38239


In [26]:
# --- Puntuación de la reseña como variable de análisis ---
df["resena_negativa"] = np.where(df["review_score"].isna(), np.nan,
                                 (df["review_score"] <= 2).astype(float))
df["resena_1_estrella"] = np.where(df["review_score"].isna(), np.nan,
                                   (df["review_score"] == 1).astype(float))

registro.anotar(
    "La puntuación de 1 a 5 no permite calcular tasas directamente",
    int(df["review_score"].notna().sum()),
    "Crear indicadores binarios de reseña negativa (1-2 estrellas) y de 1 estrella",
    "Convertidos a 0 y 1, su media es directamente el porcentaje del grupo, que es "
    "lo que se quiere medir y comparar entre segmentos.",
)

print(df["review_score"].value_counts().sort_index().to_string())

[13] La puntuación de 1 a 5 no permite calcular tasas directamente
     -> 111708 | Crear indicadores binarios de reseña negativa (1-2 estrellas) y de 1 estrella
review_score
1.0    14144
2.0     3845
3.0     9376
4.0    21205
5.0    63138


In [27]:
# --- Renombrar al español y ordenar las columnas ---
df = df.rename(columns={
    "order_id": "id_pedido", "order_item_id": "num_articulo",
    "product_id": "id_producto", "seller_id": "id_vendedor",
    "customer_id": "id_cliente", "customer_unique_id": "id_cliente_unico",
    "order_status": "estado_pedido", "order_purchase_timestamp": "fecha_compra",
    "order_approved_at": "fecha_aprobacion",
    "order_delivered_carrier_date": "fecha_recogida",
    "order_delivered_customer_date": "fecha_entrega",
    "order_estimated_delivery_date": "fecha_estimada",
    "shipping_limit_date": "fecha_limite_envio",
    "price": "precio", "freight_value": "flete",
    "customer_city": "ciudad_cliente", "customer_state": "estado_cliente",
    "customer_zip_code_prefix": "cp_cliente",
    "seller_city": "ciudad_vendedor", "seller_state": "estado_vendedor",
    "seller_zip_code_prefix": "cp_vendedor",
    "product_weight_g": "peso_g", "product_length_cm": "largo_cm",
    "product_height_cm": "alto_cm", "product_width_cm": "ancho_cm",
    "product_photos_qty": "num_fotos",
    "review_score": "nota_resena", "review_creation_date": "fecha_resena",
    "region": "region_cliente",
})

orden = [
    "id_pedido", "num_articulo", "estado_pedido",
    "fecha_compra", "anio", "mes", "anio_mes", "dia_semana", "hora_compra", "franja_horaria",
    "fecha_aprobacion", "horas_hasta_aprobacion", "fecha_recogida", "dias_hasta_envio",
    "fecha_entrega", "fecha_estimada", "dias_entrega_real", "dias_entrega_prometidos",
    "dias_retraso", "llego_tarde",
    "id_cliente", "id_cliente_unico", "cp_cliente", "ciudad_cliente", "estado_cliente",
    "estado_nombre", "region_cliente", "cliente_lat", "cliente_lng",
    "poblacion", "pib_per_capita", "tasa_pobreza", "densidad", "idh",
    "estado_lat", "estado_lng",
    "id_vendedor", "cp_vendedor", "ciudad_vendedor", "estado_vendedor", "region_vendedor",
    "vendedor_lat", "vendedor_lng", "mismo_estado", "distancia_km",
    "id_producto", "categoria", "peso_g", "largo_cm", "alto_cm", "ancho_cm",
    "volumen_cm3", "num_fotos",
    "precio", "flete", "ingreso_linea", "peso_flete_pct",
    "importe_pagado", "metodo_pago", "plazos", "num_pagos",
    "nota_resena", "resena_negativa", "resena_1_estrella", "fecha_resena",
]

faltan = set(orden) - set(df.columns)
sobran = set(df.columns) - set(orden)
print("Columnas del orden que no existen:", faltan if faltan else "ninguna")
print("Columnas que se descartan:", sobran if sobran else "ninguna")

df = df[orden]
registro.anotar(
    "Nombres de columna en inglés, mezclados y sin orden lógico",
    len(df),
    "Traducir al español, ordenar por bloques temáticos y descartar las columnas "
    "sin uso analítico",
    "Se descartan las de longitud de nombre y descripción del producto, que miden "
    "el texto del anuncio y no aportan a las preguntas de este análisis.",
)

print(f"\nTabla final: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Columnas del orden que no existen: ninguna
Columnas que se descartan: {'fecha_limite_envio'}
[14] Nombres de columna en inglés, mezclados y sin orden lógico
     -> 112650 | Traducir al español, ordenar por bloques temáticos y descartar las columnas sin uso analítico

Tabla final: 112,650 filas x 65 columnas


## 6. Verificación final

Antes de guardar hay que comprobar que la tabla es coherente.

In [29]:
print("COMPROBACIONES DE INTEGRIDAD\n")

comprobaciones = {
    "Filas conservadas frente a los artículos originales": f"{len(df):,} de {len(articulos):,}",
    "Filas duplicadas": int(df.duplicated().sum()),
    "Pedidos únicos": f"{df['id_pedido'].nunique():,}",
    "Clientes únicos": f"{df['id_cliente_unico'].nunique():,}",
    "Vendedores únicos": f"{df['id_vendedor'].nunique():,}",
    "Categorías": df["categoria"].nunique(),
    "Estados cubiertos": df["estado_cliente"].nunique(),
    "Entregas con fecha anterior a la compra": int((df["dias_entrega_real"] < 0).sum()),
    "Precios negativos o cero": int((df["precio"] <= 0).sum()),
    "Distancias imposibles (> 6.000 km)": int((df["distancia_km"] > 6000).sum()),
}

for concepto, valor in comprobaciones.items():
    print(f"  {concepto:<52} {valor}")

print(f"\nPeriodo: {df['fecha_compra'].min():%d/%m/%Y} a {df['fecha_compra'].max():%d/%m/%Y}")
print(f"Facturación total: R$ {df['ingreso_linea'].sum():,.0f}")

COMPROBACIONES DE INTEGRIDAD

  Filas conservadas frente a los artículos originales  112,650 de 112,650
  Filas duplicadas                                     0
  Pedidos únicos                                       98,666
  Clientes únicos                                      95,420
  Vendedores únicos                                    3,095
  Categorías                                           74
  Estados cubiertos                                    27
  Entregas con fecha anterior a la compra              0
  Precios negativos o cero                             0
  Distancias imposibles (> 6.000 km)                   0

Periodo: 04/09/2016 a 03/09/2018
Facturación total: R$ 15,843,553


In [30]:
ausentes = resumen_nulos(df)
print(f"Columnas con valores ausentes: {len(ausentes)} de {df.shape[1]}\n")
print(ausentes.to_string())

Columnas con valores ausentes: 27 de 65

                        nulos  porcentaje            tipo
dias_retraso             2454        2.18         float64
dias_entrega_real        2454        2.18         float64
fecha_entrega            2454        2.18  datetime64[us]
llego_tarde              2454        2.18         float64
num_fotos                1603        1.42         float64
fecha_recogida           1194        1.06  datetime64[us]
dias_hasta_envio         1194        1.06         float64
fecha_resena              942        0.84  datetime64[us]
resena_1_estrella         942        0.84         float64
resena_negativa           942        0.84         float64
nota_resena               942        0.84         float64
distancia_km              555        0.49         float64
cliente_lat               303        0.27         float64
cliente_lng               303        0.27         float64
vendedor_lat              253        0.22         float64
vendedor_lng              253  

Los ausentes que quedan **tienen explicación y se mantienen a propósito**:

- Las fechas y plazos de entrega faltan en los pedidos que no llegaron a entregarse. Rellenarlos sería inventar una entrega que no ocurrió.
- Las coordenadas y la distancia faltan en los códigos postales que no aparecen en la tabla de geolocalización, un 0,3% de las filas.
- La nota de la reseña falta en los pedidos que nadie valoró. Poner un valor medio inventaría opiniones.
- Las medidas del producto faltan en los 610 productos sin ficha completa.

En ningún caso se imputan: pandas ya excluye los nulos de los cálculos, y forzar un valor distorsionaría precisamente los análisis que vienen después.

## 7. Guardado

Además de la tabla completa se guardan las **tablas del modelo en estrella** que alimentarán PostgreSQL y Power BI. Separar dimensiones de hechos evita repetir el nombre de un estado 46.000 veces y hace que el modelo de Power BI sea mucho más ligero.

In [32]:
SALIDA = Path.cwd().parent / "data" / "processed"
SALIDA.mkdir(parents=True, exist_ok=True)

# Tabla analítica completa
df.to_csv(SALIDA / "ventas_olist_limpio.csv.gz", index=False,
          encoding="utf-8-sig", compression="gzip")

# Registro de limpieza
registro_df = registro.a_dataframe()
registro_df.to_csv(SALIDA / "registro_limpieza.csv", index=False, encoding="utf-8-sig")

tamano = (SALIDA / "ventas_olist_limpio.csv.gz").stat().st_size / 1e6
print(f"ventas_olist_limpio.csv.gz -> {len(df):,} filas x {df.shape[1]} columnas ({tamano:.1f} MB)")
print(f"registro_limpieza.csv    -> {len(registro_df)} pasos documentados")

ventas_olist_limpio.csv.gz -> 112,650 filas x 65 columnas (25.2 MB)
registro_limpieza.csv    -> 14 pasos documentados


In [33]:
# --- Modelo en estrella ---
dim_cliente = (df[["id_cliente_unico", "ciudad_cliente", "estado_cliente",
                   "cliente_lat", "cliente_lng"]]
               .drop_duplicates("id_cliente_unico").reset_index(drop=True))

dim_vendedor = (df[["id_vendedor", "ciudad_vendedor", "estado_vendedor",
                    "region_vendedor", "vendedor_lat", "vendedor_lng"]]
                .drop_duplicates("id_vendedor").reset_index(drop=True))

dim_producto = (df[["id_producto", "categoria", "peso_g", "largo_cm", "alto_cm",
                    "ancho_cm", "volumen_cm3", "num_fotos"]]
                .drop_duplicates("id_producto").reset_index(drop=True))

dim_estado = estados_limpios.copy()

# Dimensión de calendario: una fila por día del periodo
fechas = pd.date_range(df["fecha_compra"].min().normalize(),
                       df["fecha_compra"].max().normalize(), freq="D")
dim_fecha = pd.DataFrame({"fecha": fechas})
dim_fecha["anio"] = dim_fecha["fecha"].dt.year
dim_fecha["mes"] = dim_fecha["fecha"].dt.month
dim_fecha["nombre_mes"] = dim_fecha["fecha"].dt.month_name()
dim_fecha["trimestre"] = dim_fecha["fecha"].dt.quarter
dim_fecha["dia_semana"] = dim_fecha["fecha"].dt.day_name()
dim_fecha["es_fin_de_semana"] = dim_fecha["fecha"].dt.dayofweek >= 5
dim_fecha["anio_mes"] = dim_fecha["fecha"].dt.to_period("M").astype(str)

# Tabla de hechos: solo claves y medidas
hechos = df[["id_pedido", "num_articulo", "fecha_compra", "id_cliente_unico",
             "id_vendedor", "id_producto", "estado_cliente", "estado_pedido",
             "precio", "flete", "ingreso_linea", "peso_flete_pct",
             "dias_entrega_real", "dias_entrega_prometidos", "dias_retraso",
             "llego_tarde", "distancia_km", "mismo_estado", "nota_resena",
             "resena_negativa", "resena_1_estrella", "metodo_pago", "plazos",
             "importe_pagado", "hora_compra", "franja_horaria"]].copy()
hechos["fecha"] = hechos["fecha_compra"].dt.normalize()

modelo = {
    "hechos_ventas": hechos,
    "dim_cliente": dim_cliente,
    "dim_vendedor": dim_vendedor,
    "dim_producto": dim_producto,
    "dim_estado": dim_estado,
    "dim_fecha": dim_fecha,
}

carpeta_modelo = SALIDA / "modelo_estrella"
carpeta_modelo.mkdir(exist_ok=True)

for nombre, tabla in modelo.items():
    tabla.to_csv(carpeta_modelo / f"{nombre}.csv.gz", index=False,
                 encoding="utf-8-sig", compression="gzip")
    print(f"{nombre:<18} {tabla.shape[0]:>7,} filas x {tabla.shape[1]:>2} columnas")

hechos_ventas      112,650 filas x 27 columnas
dim_cliente         95,420 filas x  5 columnas
dim_vendedor         3,095 filas x  6 columnas
dim_producto        32,951 filas x  8 columnas
dim_estado              27 filas x 10 columnas
dim_fecha              730 filas x  8 columnas


In [34]:
registro_df

### Resumen

De las **112.650 líneas de pedido** originales se conservan las 112.650: no se ha eliminado ni un registro. La tabla final tiene **64 columnas**, de las que 24 son derivadas y 10 provienen de la segunda fuente.

Los pasos documentados quedan en `registro_limpieza.csv`, y el modelo en estrella en `data/processed/modelo_estrella/` listo para cargar en PostgreSQL.

Todas las salidas van comprimidas en gzip: `pandas` las lee igual que un CSV normal (`pd.read_csv` detecta la compresión por la extensión) y el repositorio pasa de 75 MB a unos 12.

El análisis continúa en `02_analisis_y_estadistica.ipynb`.